# Text-to-Speech (TTS)

Due to security considerations, it's not feasible to directly access audio devices through JupyterLab because of the environment's limitations. Therefore, the code blocks provided here are not intended for execution by users.

The programs presented here originate from the product's main program file, audio_ctrl.py. You can refer to these code snippets to gain insight into how the product's main program facilitates text-to-speech functionality.转语音功能的。

In [ ]:
import pyttsx3  # Import pyttsx3 library for local TTS (text-to-speech)
import threading  # Import threading module for multithreading
import numpy as np  # Import NumPy for audio data processing
import sherpa_onnx  # Import sherpa_onnx to load offline TTS models
import soundfile as sf  # Import soundfile to save audio files
import re  # Import regex library to detect Chinese characters in text

# Initialize pyttsx3 engine (used to play non-Chinese TTS)
engine = pyttsx3.init()

# Create a threading event to control speech playback synchronization and prevent overlapping audio
play_audio_event = threading.Event()

# Set TTS playback speed (180 means slightly faster speech rate)
engine.setProperty('rate', 180)

# Configure sherpa-onnx offline TTS model (Chinese VITS model)
tts_config = sherpa_onnx.OfflineTtsConfig(
    model=sherpa_onnx.OfflineTtsModelConfig(
        vits=sherpa_onnx.OfflineTtsVitsModelConfig(
            model='./models/sherpa-onnx-vits-zh-ll/model.onnx',  # Model file
            lexicon='./models/sherpa-onnx-vits-zh-ll/lexicon.txt',  # Lexicon file
            data_dir='',
            dict_dir='./models/sherpa-onnx-vits-zh-ll/dict',  # Pinyin dictionary
            tokens='./models/sherpa-onnx-vits-zh-ll/tokens.txt',  # Tokens file
        ),
        matcha=sherpa_onnx.OfflineTtsMatchaModelConfig(),  # Placeholder for other TTS models
        kokoro=sherpa_onnx.OfflineTtsKokoroModelConfig(),  # Placeholder for other TTS models
        provider='cuda',  # Use GPU for inference
        debug=False,
        num_threads=2,  # Number of inference threads
    ),
    rule_fsts='./models/sherpa-onnx-vits-zh-ll/number.fst',  # Number transcription rules
    max_num_sentences=1,  # Process one sentence at a time
)

# Initialize offline TTS engine
tts = sherpa_onnx.OfflineTts(tts_config)


def play_audio(input_audio_file):
    """Play a specified audio file."""
    if not usb_connected:
        return
    try:
        pygame.mixer.music.load(input_audio_file)  # Load audio file
        pygame.mixer.music.play()  # Play audio
    except:
        play_audio_event.clear()
        return
    while pygame.mixer.music.get_busy():  # Wait until playback finishes
        pass
    time.sleep(min_time_bewteen_play)  # Playback interval delay
    play_audio_event.clear()


def contains_chinese(text):
    """Check if the text contains any Chinese characters."""
    return bool(re.search('[\u4e00-\u9fff]', text))


def play_speech(input_text):
    """Convert text to speech and play."""
    filename = 'audio-say.wav'
    if not usb_connected:
        return

    try:
        if contains_chinese(input_text):
            # Use VITS model for Chinese text
            audio = tts.generate(input_text, sid=4, speed=1.0)
            scale = 4  # Amplify volume
            samples = np.array(audio.samples) * scale
            samples = np.clip(samples, -1.0, 1.0)  # Clamp volume range

            # Save to WAV file
            sf.write(filename, samples, samplerate=audio.sample_rate, subtype="PCM_16")

            # Wait for file to be generated
            for _ in range(10):
                if os.path.exists(filename) and os.path.getsize(filename) > 0:
                    break
                time.sleep(0.1)

            play_audio(filename)  # Play the audio file
        else:
            # For non-Chinese text, directly use pyttsx3 to play
            engine.say(input_text)
            engine.runAndWait()

    except Exception as e:
        print(f"[play failure] {e}")
    finally:
        # Delete temporary file after playback
        if os.path.exists(filename):
            try:
                os.remove(filename)
            except Exception as e:
                print(f"[delete file failure] {e}")
        play_audio_event.clear()


def play_speech_thread(input_text):
    """Play speech in a new thread to avoid blocking the main thread."""
    if play_audio_event.is_set():  # Ignore if playback is ongoing
        return
    play_audio_event.set()  # Mark playback started
    speech_thread = threading.Thread(target=play_speech, args=(input_text,))
    speech_thread.start()  # Start new thread

This code uses the sherpa_onnx and pyttsx3 libraries to implement text-to-speech functionality, and uses the threading module to create a thread for asynchronous speech playback. The play_speech() function is used to play the speech of a specified text in the main thread, while the play_speech_thread() function is used to play speech in a new thread to avoid blocking the main thread. At the same time, speech playback synchronization is controlled through play_audio_event to ensure that only one speech is played at a time.